In [1]:
import argparse

import ase
from ase import Atoms
import numpy as np
from pathlib import Path
from glob import glob
from tqdm import tqdm
from datetime import datetime
from omegaconf import DictConfig, OmegaConf

import torch
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

from spec2struct.diffusion.diffusion_cfg import CSPDiffusion
from spec2struct.utils.constants import cdvae_train_num_elements_distribution
from spec2struct.utils.utils import decode
from spec2struct.dataset.datamodule import CrystalDataModule, worker_init_fn
from spec2struct.dataset.dataset import CrystalDataset

In [2]:
root_path = "outputs/2d_dos_ft_pretrained_perov5supercell"

In [3]:
root_path = Path(root_path)

now = datetime.now()
formatted_time = now.strftime("%d%m%Y_%H%M%S")

# load config
print("Loading model...")
config_path = root_path / 'hparams.yaml'
config = OmegaConf.load(config_path)

# load checkpoint
ckpt_path = glob(str(root_path / '*.ckpt'))
if len(ckpt_path) == 0:
    raise ValueError("No checkpoint file found.")
elif len(ckpt_path) > 1:
    raise ValueError("Multiple checkpoint files found.")
ckpt_path = ckpt_path[0]

model = CSPDiffusion.load_from_checkpoint(ckpt_path, config=config)
model.to('cuda')

Loading model...


CSPDiffusion(
  (decoder): CSPNet(
    (node_embedding): Linear(in_features=100, out_features=512, bias=True)
    (atom_latent_emb): Linear(in_features=768, out_features=512, bias=True)
    (act_fn): SiLU()
    (dis_emb): SinusoidsEmbedding()
    (csp_layer_0): CSPLayer(
      (act_fn): SiLU()
      (dis_emb): SinusoidsEmbedding()
      (edge_mlp): Sequential(
        (0): Linear(in_features=1801, out_features=512, bias=True)
        (1): SiLU()
        (2): Linear(in_features=512, out_features=512, bias=True)
        (3): SiLU()
      )
      (node_mlp): Sequential(
        (0): Linear(in_features=1024, out_features=512, bias=True)
        (1): SiLU()
        (2): Linear(in_features=512, out_features=512, bias=True)
        (3): SiLU()
      )
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    )
    (csp_layer_1): CSPLayer(
      (act_fn): SiLU()
      (dis_emb): SinusoidsEmbedding()
      (edge_mlp): Sequential(
        (0): Linear(in_features=1801, out_fea

In [5]:
model.decoder.cfg

False